In [89]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
class MyHashMap(object):
    def __init__(self):
        self.size = 100
        self.arr = [[]] * self.size

    def get(self, key: str):
        assert type(key) is str, "key must be a string"
        key_int = sum([ord(c) for c in key])
        bucket = self.arr[key_int % self.size]
        for k, v in bucket:
            if k == key:
                return v
        return None

    def set(self, key: str, value):
        assert type(key) is str, "key must be a string"
        key_int = sum([ord(c) for c in key])
        bucket = self.arr[key_int % self.size]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket[i] = (key, value)
                return
        bucket.append((key, value))

    def delete(self, key: str):
        assert type(key) is str, "key must be a string"
        key_int = sum([ord(c) for c in key])
        bucket = self.arr[key_int % self.size]
        for i, (k, v) in enumerate(bucket):
            if k == key:
                bucket.pop(i)
                return

map = MyHashMap()
map.set("a", 123)
map.set("test", 123)
assert map.get("test") == 123
assert map.get("a") == 123
assert map.get("b") is None
map.set("a", "re-wrtie")
assert map.get("a") == "re-wrtie"
map.delete("a")
assert map.get("a") is None

In [4]:
class MyHeap(object):
    def __init__(self):
        self.arr = []

    def push(self, val):
        self.arr.append(val)
        self.heapify(len(self.arr)-1)

    def heapify(self, idx):
        if idx == 0:
            return

        parent = self.arr[(idx-1)//2]
        if parent > self.arr[idx]:
            self.arr[(idx-1)//2], self.arr[idx] = self.arr[idx], self.arr[(idx-1)//2]
            self.heapify((idx-1)//2)

    def heapifydown(self, idx):
        left = idx * 2 + 1
        right = idx * 2 + 2
        min_idx = idx

        if left < len(self.arr) and self.arr[left] < self.arr[idx]:
            min_idx = left
        
        if right < len(self.arr) and self.arr[right] < self.arr[min_idx]:
            min_idx = right

        if min_idx != idx:
            self.arr[min_idx], self.arr[idx] = self.arr[idx], self.arr[min_idx]
            self.heapifydown(min_idx)        

    def peek(self):
        if len(self.arr) == 0:
            return None

        return self.arr[0]        

    def pop(self):
        if len(self.arr) == 0:
            return None
        
        if len(self.arr) == 1:
            return self.arr.pop()

        val = self.arr[0]
        self.arr[0] = self.arr.pop()
        self.heapifydown(0)        
        return val
    
    def __len__(self):
        return len(self.arr)
    
heap = MyHeap()
heap.push(4)
heap.push(6)
heap.push(1)
heap.push(1)
heap.push(100)
assert heap.peek() == 1
assert heap.pop() == 1
assert heap.peek() == 1
heap.pop()
assert len(heap) == 3
assert heap.peek() == 4
heap.push(-100)
heap.push(10000)
assert heap.peek() == -100
heap.pop()
heap.pop()
heap.pop()
heap.pop()
heap.pop()
assert heap.peek() is None

# VAR of Portfolios

In [7]:
# Parameters
S0 = [100, 150, 200]
w = [0.4, 0.3, 0.3]
mu = [0.1, 0.12, 0.15]
sigma = [0.2, 0.25, 0.3]
corr = np.array([[1, 0.5, 0.3], [0.5, 1, 0.4], [0.3, 0.4, 1]])
T = 1 / (6.5 * 252)  # 1 hour
M = 100000

In [103]:
def portfolio_var(S0, w, mu, sigma, corr, T, M, alpha=0.95):

    # convert to array
    S0 = np.array(S0)
    w = np.array(w)
    mu = np.array(mu)
    sigma = np.array(sigma)
    corr = np.array(corr)

    # convert correlation matrix to covariance
    cov = np.outer(sigma, sigma) * corr

    # drift term    
    drift = np.array(mu) - 0.5 * np.array(sigma) ** 2

    # diffusion term
    # L = np.linalg.cholesky(T*cov)
    # Z = np.random.normal(size=(M, len(S0)))
    # diff = np.dot(Z, L.T)
    diff = np.random.multivariate_normal(np.zeros(len(S0)), cov * T, size=(M, len(S0)))
    ret = drift + diff

    # return for 1 period 
    S_T = S0 * np.exp(ret)

    # portfolio
    V0 = S0 * w
    V_T = S_T * w
    V_ret = V_T - V0

    # var
    var = np.percentile(V_ret, (1-alpha)*100)
    return var

portfolio_var(S0, w, mu, sigma, corr, T, M, 0.95)

3.111848751283208